# Midstream PDM — ML Training Pipeline (v7)

## Required Packages

| Package | Purpose |
|---------|--------|
| `snowflake-snowpark-python` | Snowflake connectivity |
| `snowflake-ml-python` | Model Registry |
| `xgboost` | Classifier + Regressor |
| `scikit-learn` | Preprocessing, metrics |
| `imbalanced-learn` | SMOTE for class balancing |
| `pandas`, `numpy` | Data manipulation |

**Pipeline**: Feature Store → 31 Features → SMOTE → XGBoost → ML Registry

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, mean_absolute_error, r2_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier, XGBRegressor

from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

session = get_active_session()
session.sql('USE DATABASE PDM_DEMO').collect()
print('Connected')

## Step 1: Load Feature Store
The Feature Store is pre-built from raw hourly telemetry via 24-hour rolling windows. Each row = one asset on one day.

In [ ]:
pdf = session.sql('SELECT * FROM ANALYTICS.FEATURE_STORE ORDER BY ASSET_ID, AS_OF_TS').to_pandas()
print(f'Loaded {len(pdf):,} rows')

## Step 2: Engineer 31 Features
Derived features include z-score severity metrics, interaction terms, and a composite degradation intensity score. These are what give the model signal beyond raw sensor averages.

In [ ]:
FEATURE_COLS = [
    'VIBRATION_MEAN_24H', 'VIBRATION_STD_24H', 'VIBRATION_MAX_24H', 'VIBRATION_TREND',
    'TEMPERATURE_MEAN_24H', 'TEMPERATURE_STD_24H', 'TEMPERATURE_MAX_24H', 'TEMPERATURE_TREND',
    'PRESSURE_MEAN_24H', 'PRESSURE_STD_24H', 'FLOW_RATE_MEAN_24H',
    'RPM_MEAN_24H', 'RPM_STD_24H', 'POWER_DRAW_MEAN_24H',
    'DAYS_SINCE_MAINTENANCE', 'MAINTENANCE_COUNT_90D', 'OPERATING_HOURS',
    'DIFF_PRESSURE_MEAN_24H', 'SEAL_TEMP_MEAN_24H',
    'DISCHARGE_TEMP_MEAN_24H', 'COMPRESSION_RATIO_MEAN', 'OIL_PRESSURE_MEAN_24H',
    'IS_PUMP', 'VIB_TEMP_INTERACTION', 'POWER_EFFICIENCY', 'PRESSURE_VARIABILITY',
    'VIB_DEVIATION', 'TEMP_DEVIATION', 'MAINT_RECENCY_SCORE',
    'VIB_SEVERITY', 'TEMP_SEVERITY', 'DEGRADATION_INTENSITY',
]

for col in pdf.columns:
    if pdf[col].dtype in ['float64', 'int64']:
        pdf[col] = pdf[col].fillna(0)

pdf['IS_PUMP'] = (pdf['ASSET_TYPE'] == 'PUMP').astype(int)
pdf['VIB_TEMP_INTERACTION'] = pdf['VIBRATION_MEAN_24H'] * pdf['TEMPERATURE_MEAN_24H'] / 1000
pdf['POWER_EFFICIENCY'] = pdf['POWER_DRAW_MEAN_24H'] / pdf['FLOW_RATE_MEAN_24H'].clip(lower=1)
pdf['PRESSURE_VARIABILITY'] = pdf['PRESSURE_STD_24H'] / pdf['PRESSURE_MEAN_24H'].clip(lower=1)
pdf['VIB_DEVIATION'] = pdf['VIBRATION_MAX_24H'] - pdf['VIBRATION_MEAN_24H']
pdf['TEMP_DEVIATION'] = pdf['TEMPERATURE_MAX_24H'] - pdf['TEMPERATURE_MEAN_24H']
pdf['MAINT_RECENCY_SCORE'] = 1 / pdf['DAYS_SINCE_MAINTENANCE'].clip(lower=1)

normal = pdf['FAILURE_LABEL'] == 'NORMAL'
VIB_MEAN, VIB_STD = pdf.loc[normal, 'VIBRATION_MEAN_24H'].mean(), max(pdf.loc[normal, 'VIBRATION_MEAN_24H'].std(), 0.1)
TEMP_MEAN, TEMP_STD = pdf.loc[normal, 'TEMPERATURE_MEAN_24H'].mean(), max(pdf.loc[normal, 'TEMPERATURE_MEAN_24H'].std(), 0.1)

pdf['VIB_SEVERITY'] = (pdf['VIBRATION_MEAN_24H'] - VIB_MEAN) / VIB_STD
pdf['TEMP_SEVERITY'] = (pdf['TEMPERATURE_MEAN_24H'] - TEMP_MEAN) / TEMP_STD
pdf['DEGRADATION_INTENSITY'] = np.sqrt(pdf['VIB_SEVERITY'].clip(lower=0)**2 + pdf['TEMP_SEVERITY'].clip(lower=0)**2)

for col in FEATURE_COLS:
    if col in pdf.columns:
        pdf[col] = pdf[col].replace([np.inf, -np.inf], 0).fillna(0)
print(f'{len(FEATURE_COLS)} features engineered')

## Step 3: Train/Test Split + SMOTE Oversampling
80/20 stratified split. SMOTE balances the minority failure classes (BEARING_WEAR, SEAL_LEAK, etc.) so the model doesn't just predict NORMAL for everything.

In [ ]:
le = LabelEncoder()
pdf['LABEL_ENCODED'] = le.fit_transform(pdf['FAILURE_LABEL'])

X = pdf[FEATURE_COLS].values
y_cls = pdf['LABEL_ENCODED'].values
y_rul = pdf['DAYS_TO_FAILURE'].values

X_train, X_test, y_train_cls, y_test_cls, y_train_rul, y_test_rul = train_test_split(
    X, y_cls, y_rul, test_size=0.2, random_state=42, stratify=y_cls
)

smote = SMOTE(random_state=42, k_neighbors=2)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train_cls)
print(f'After SMOTE: {X_train_bal.shape[0]:,}')

## Step 4: XGBoost Failure Classifier
Multi-class classification: NORMAL, BEARING_WEAR, VALVE_FAILURE, SEAL_LEAK, SURGE, OVERHEATING. Outputs probability distributions across all classes.

In [ ]:
clf = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=3.0,
    objective='multi:softprob', num_class=len(le.classes_),
    random_state=42, tree_method='hist'
)
clf.fit(X_train_bal, y_train_bal)
y_pred = clf.predict(X_test)
f1 = f1_score(y_test_cls, y_pred, average='macro')
acc = accuracy_score(y_test_cls, y_pred)
print(f'Classifier: F1={f1:.4f}, Accuracy={acc:.4f}')

## Step 5: XGBoost RUL Regressor
Remaining Useful Life regression — trained only on non-NORMAL samples. Predicts days until failure for assets showing degradation.

In [ ]:
normal_idx = le.transform(['NORMAL'])[0]
nn_train = y_train_cls != normal_idx
nn_test = y_test_cls != normal_idx

reg = XGBRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=2.0, random_state=42, tree_method='hist'
)
reg.fit(X_train[nn_train], y_train_rul[nn_train])
y_pred_rul = reg.predict(X_test[nn_test])
mae = mean_absolute_error(y_test_rul[nn_test], y_pred_rul)
r2 = r2_score(y_test_rul[nn_test], y_pred_rul)
print(f'RUL Regressor: MAE={mae:.2f}d, R2={r2:.4f}')

## Step 6: Register Models to Snowflake ML Registry
Both models are versioned in `PDM_DEMO.ML` with metrics (F1, accuracy, MAE, R²). The scoring notebook and SPCS app load models directly from this registry.

In [ ]:
MODEL_VERSION = 'v7'
registry = Registry(session=session, database_name='PDM_DEMO', schema_name='ML')
X_df = pd.DataFrame(X_test, columns=FEATURE_COLS)

clf_mv = registry.log_model(
    model=clf, model_name='FAILURE_CLASSIFIER', version_name=MODEL_VERSION,
    sample_input_data=X_df.head(10), conda_dependencies=['xgboost', 'scikit-learn'],
    metrics={'f1_macro': round(float(f1), 4), 'accuracy': round(float(acc), 4)},
)
print(f'FAILURE_CLASSIFIER {MODEL_VERSION} registered')

reg_mv = registry.log_model(
    model=reg, model_name='RUL_REGRESSOR', version_name=MODEL_VERSION,
    sample_input_data=X_df.head(10), conda_dependencies=['xgboost', 'scikit-learn'],
    metrics={'mae_days': round(float(mae), 2), 'r2': round(float(r2), 4)},
)
print(f'RUL_REGRESSOR {MODEL_VERSION} registered')

## Step 7: Verify Registered Models
List all models and versions in the registry to confirm successful registration.

In [ ]:
print('Models in PDM_DEMO.ML:')
for m in registry.models():
    print(f'  {m.name}')
    for v in m.versions():
        print(f'    - {v.version_name}')